# 02 — Supervised Classification

Wallet classifier: Logistic Regression → KNN → SVM → Random Forest → XGBoost.

Multi-class: exchange, dex_trader, dex_lp, nft_collector, bot, phishing, normal.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

In [ ]:
# Synthetic dataset (ganti dengan labeled wallet dataset)
from sklearn.datasets import make_classification
n_classes = 7
X, y = make_classification(n_samples=2000, n_features=30, n_informative=18, n_classes=n_classes, random_state=42)
class_names = ['exchange','dex_trader','dex_lp','nft_collector','bot','phishing','normal']
y_str = np.array(class_names)[y]
print(f'Data: {X.shape}, class distribution:')
print(pd.Series(y_str).value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_str, test_size=0.2, stratify=y_str, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, multi_class='multinomial'),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'SVM (rbf)': SVC(kernel='rbf', probability=True),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=12),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    cv_score = cross_val_score(model, X_train_s, y_train_enc, cv=cv, scoring='f1_macro').mean()
    model.fit(X_train_s, y_train_enc)
    y_pred = model.predict(X_test_s)
    f1 = f1_score(y_test_enc, y_pred, average='macro')
    results[name] = {'cv_mean': cv_score, 'test_f1': f1}
    print(f'{name:20s} | CV F1: {cv_score:.3f} | Test F1: {f1:.3f}')

In [ ]:
# Best model confusion matrix
best_name = max(results, key=lambda k: results[k]['test_f1'])
best_model = models[best_name]
cm = confusion_matrix(y_test_enc, best_model.predict(X_test_s))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'{best_name} — Confusion Matrix'); plt.xticks(rotation=45); plt.show()

In [ ]:
print(classification_report(y_test_enc, best_model.predict(X_test_s), target_names=le.classes_))